In [1]:
import random
import pandas as pd
import json

random.seed(706)
n = 400
lokasi_list = ["Cabang Magelang", "Cabang Semarang", "Cabang Yogyakarta"]
kategori_dirty = ["tarik tunai","Tarik Tunai","TARIK TUNAI","setor","Setor","transfer","Transfer"]
teller_ids = [f"TL{i:03d}" for i in range(1, 21)]
lokasi_per_teller = {tid: random.choice(lokasi_list) for tid in teller_ids}

rows = []
for i in range(n):
    teller_id = random.choice(teller_ids)
    rows.append({
        "transaksi_id": f"TRX-{1000+i}",
        "teller_id": teller_id,
        "cabang": lokasi_per_teller[teller_id],
        "jenis_transaksi": random.choice(kategori_dirty),
        "waktu_layanan_menit": random.randint(3, 25),
        "biaya_administrasi": random.choice(
            [2000,3000,5000,7500,10000]),
        "status_transaksi": random.choice(
            ["Selesai","Selesai","Selesai","Dibatalkan"]),
    })
    
rows += [rows[5], rows[22], rows[85]]
for idx in [7, 42, 145, 185]:
    rows[idx] = dict(rows[idx])
    rows[idx]["waktu_layanan_menit"] = None
for idx in [11, 95]:
    rows[idx] = dict(rows[idx])
    rows[idx]["biaya_administrasi"] = None
for idx in [225, 275]:
    rows[idx] = dict(rows[idx])
    rows[idx]["waktu_layanan_menit"] = -random.randint(1,10)
    
pd.DataFrame(rows).to_csv("transaksi.csv", index=False)
teller_master = [
    {"teller_id": tid, "nama_teller": f"Teller-{tid}",
    "cabang": lokasi_per_teller[tid],
    "rating_teller": round(random.uniform(3.0,5.0),1)}
    for tid in teller_ids
]

with open("teller.json", "w") as f:
    for t in teller_master:
        f.write(json.dumps(t) + "\n")
        
print("Data berhasil dibuat: transaksi.csv dan teller.json")

Data berhasil dibuat: transaksi.csv dan teller.json


In [3]:
# Membuat direktori baru di hdfs dan upload transaksi.csv ke sana
!hdfs dfs -mkdir -p /user/mazedev/latihan/uts/raw
!hdfs dfs -put -f transaksi.csv /user/mazedev/latihan/uts/raw/

#Verifikasi berkas
!hdfs dfs -ls /user/mazedev/latihan/uts/raw

Found 1 items
-rw-r--r--   1 mazedev supergroup      24991 2026-10-07 22:38 /user/mazedev/latihan/uts/raw/transaksi.csv


In [4]:
# Membuat Sparksessin
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("LatihanResponsi").master("local[*]").getOrCreate()

print("SparkSession berhasil dibuat:", spark.version)

26/10/07 22:42:01 WARN Utils: Your hostname, mazedev-1701 resolves to a loopback address: 127.0.1.1; using 192.168.100.6 instead (on interface wlp2s0)
26/10/07 22:42:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/07 22:42:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil dibuat: 3.5.9


In [6]:
# Membaca transaksi.csv dari HDFS
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mazedev/latihan/uts/raw", header=True, inferSchema=True)
df_transaksi.printSchema()

root
 |-- transaksi_id: string (nullable = true)
 |-- teller_id: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- jenis_transaksi: string (nullable = true)
 |-- waktu_layanan_menit: double (nullable = true)
 |-- biaya_administrasi: double (nullable = true)
 |-- status_transaksi: string (nullable = true)



In [7]:
# Baca teller.json
df_teller = spark.read.json("teller.json")
df_teller.printSchema()

root
 |-- cabang: string (nullable = true)
 |-- nama_teller: string (nullable = true)
 |-- rating_teller: double (nullable = true)
 |-- teller_id: string (nullable = true)



In [9]:
# Menghitung jumlah baris kedua dataFrame
print("Jumlah baris df_transaksi:", df_transaksi.count())
print("Jumlah baris df_teller:", df_teller.count())

Jumlah baris df_transaksi: 403
Jumlah baris df_teller: 20


In [10]:
# Eksplorasi dan pembersihan data
df_transaksi.show()

+------------+---------+-----------------+---------------+-------------------+------------------+----------------+
|transaksi_id|teller_id|           cabang|jenis_transaksi|waktu_layanan_menit|biaya_administrasi|status_transaksi|
+------------+---------+-----------------+---------------+-------------------+------------------+----------------+
|    TRX-1000|    TL008|Cabang Yogyakarta|       Transfer|                4.0|            2000.0|         Selesai|
|    TRX-1001|    TL013|Cabang Yogyakarta|    tarik tunai|               16.0|            5000.0|         Selesai|
|    TRX-1002|    TL007|Cabang Yogyakarta|          Setor|               15.0|           10000.0|      Dibatalkan|
|    TRX-1003|    TL006|Cabang Yogyakarta|       transfer|               21.0|            2000.0|         Selesai|
|    TRX-1004|    TL009|  Cabang Semarang|          Setor|               11.0|            2000.0|         Selesai|
|    TRX-1005|    TL004|Cabang Yogyakarta|          Setor|               10.0|  

In [17]:
from pyspark.sql.functions import lower

# menyelaraskan huruf kecil di jenis_transaksi
df_transaksi = df_transaksi.withColumn("jenis_transaksi", lower(df_transaksi["jenis_transaksi"]))

df_transaksi.show()

+------------+---------+-----------------+---------------+-------------------+------------------+----------------+
|transaksi_id|teller_id|           cabang|jenis_transaksi|waktu_layanan_menit|biaya_administrasi|status_transaksi|
+------------+---------+-----------------+---------------+-------------------+------------------+----------------+
|    TRX-1000|    TL008|Cabang Yogyakarta|       transfer|                4.0|            2000.0|         Selesai|
|    TRX-1001|    TL013|Cabang Yogyakarta|    tarik tunai|               16.0|            5000.0|         Selesai|
|    TRX-1002|    TL007|Cabang Yogyakarta|          setor|               15.0|           10000.0|      Dibatalkan|
|    TRX-1003|    TL006|Cabang Yogyakarta|       transfer|               21.0|            2000.0|         Selesai|
|    TRX-1004|    TL009|  Cabang Semarang|          setor|               11.0|            2000.0|         Selesai|
|    TRX-1005|    TL004|Cabang Yogyakarta|          setor|               10.0|  

In [22]:
# Mengisi value Null di waktu_layanan_menit dan biaya_administrasi menjadi 0.0
df_transaksi = df_transaksi.na.fill(0, subset=["waktu_layanan_menit", "biaya_administrasi"])

df_transaksi.show()

+------------+---------+-----------------+---------------+-------------------+------------------+----------------+
|transaksi_id|teller_id|           cabang|jenis_transaksi|waktu_layanan_menit|biaya_administrasi|status_transaksi|
+------------+---------+-----------------+---------------+-------------------+------------------+----------------+
|    TRX-1000|    TL008|Cabang Yogyakarta|       transfer|                4.0|            2000.0|         Selesai|
|    TRX-1001|    TL013|Cabang Yogyakarta|    tarik tunai|               16.0|            5000.0|         Selesai|
|    TRX-1002|    TL007|Cabang Yogyakarta|          setor|               15.0|           10000.0|      Dibatalkan|
|    TRX-1003|    TL006|Cabang Yogyakarta|       transfer|               21.0|            2000.0|         Selesai|
|    TRX-1004|    TL009|  Cabang Semarang|          setor|               11.0|            2000.0|         Selesai|
|    TRX-1005|    TL004|Cabang Yogyakarta|          setor|               10.0|  

In [39]:
# Transformasi Data dan Penggabungan
df_gabungan = df_transaksi.join(df_teller.drop("cabang"), on="teller_id", how="left")
df_gabungan.show()

+---------+------------+-----------------+---------------+-------------------+------------------+----------------+------------+-------------+
|teller_id|transaksi_id|           cabang|jenis_transaksi|waktu_layanan_menit|biaya_administrasi|status_transaksi| nama_teller|rating_teller|
+---------+------------+-----------------+---------------+-------------------+------------------+----------------+------------+-------------+
|    TL008|    TRX-1000|Cabang Yogyakarta|       transfer|                4.0|            2000.0|         Selesai|Teller-TL008|          3.0|
|    TL013|    TRX-1001|Cabang Yogyakarta|    tarik tunai|               16.0|            5000.0|         Selesai|Teller-TL013|          3.8|
|    TL007|    TRX-1002|Cabang Yogyakarta|          setor|               15.0|           10000.0|      Dibatalkan|Teller-TL007|          3.5|
|    TL006|    TRX-1003|Cabang Yogyakarta|       transfer|               21.0|            2000.0|         Selesai|Teller-TL006|          4.0|
|    T

In [40]:
from pyspark.sql.functions import when, col

df_gabungan = df_gabungan.withColumn(
    "kategori_sla",
    when(col("waktu_layanan_menit") <= 15, "Sesuai SLA").otherwise("Melebihi SLA")
)

df_gabungan.show(10)

+---------+------------+-----------------+---------------+-------------------+------------------+----------------+------------+-------------+------------+
|teller_id|transaksi_id|           cabang|jenis_transaksi|waktu_layanan_menit|biaya_administrasi|status_transaksi| nama_teller|rating_teller|kategori_sla|
+---------+------------+-----------------+---------------+-------------------+------------------+----------------+------------+-------------+------------+
|    TL008|    TRX-1000|Cabang Yogyakarta|       transfer|                4.0|            2000.0|         Selesai|Teller-TL008|          3.0|  Sesuai SLA|
|    TL013|    TRX-1001|Cabang Yogyakarta|    tarik tunai|               16.0|            5000.0|         Selesai|Teller-TL013|          3.8|Melebihi SLA|
|    TL007|    TRX-1002|Cabang Yogyakarta|          setor|               15.0|           10000.0|      Dibatalkan|Teller-TL007|          3.5|  Sesuai SLA|
|    TL006|    TRX-1003|Cabang Yogyakarta|       transfer|            

In [44]:
# Analisis dengan window function
from pyspark.sql.functions import rank, row_number, count
from pyspark.sql.window import Window
# Top 3 teller per Cabang

df_sla = df_gabungan.filter(col("kategori_sla") == "Sesuai SLA") \
    .groupBy("cabang", "nama_teller") \
    .agg(count("transaksi_id").alias("jumlah"))

window_a = Window.partitionBy(col("cabang")).orderBy(col("jumlah").desc())

top_3 = df_sla.withColumn("ranking", rank().over(window_a)) \
    .filter(col("ranking") < 4) \
    .select("cabang", "nama_teller", "jumlah")

top_3.show()

+-----------------+------------+------+
|           cabang| nama_teller|jumlah|
+-----------------+------------+------+
|  Cabang Magelang|Teller-TL010|    16|
|  Cabang Magelang|Teller-TL019|    11|
|  Cabang Magelang|Teller-TL015|    11|
|  Cabang Semarang|Teller-TL017|    17|
|  Cabang Semarang|Teller-TL016|    16|
|  Cabang Semarang|Teller-TL014|    15|
|Cabang Yogyakarta|Teller-TL006|    20|
|Cabang Yogyakarta|Teller-TL011|    14|
|Cabang Yogyakarta|Teller-TL012|    13|
+-----------------+------------+------+



In [47]:
# Best Teller per cabang
window_b = Window.partitionBy(col("cabang")).orderBy(col("rating_teller").desc())

best_teller = df_gabungan.withColumn("ranking", row_number().over(window_b)).filter(col("ranking") < 2).select("cabang", "nama_teller", "rating_teller")

best_teller.show()

+-----------------+------------+-------------+
|           cabang| nama_teller|rating_teller|
+-----------------+------------+-------------+
|  Cabang Magelang|Teller-TL015|          4.2|
|  Cabang Semarang|Teller-TL009|          4.8|
|Cabang Yogyakarta|Teller-TL005|          4.9|
+-----------------+------------+-------------+



In [50]:
# Analisis dengan Spark SQL
df_gabungan.createOrReplaceTempView("gabungan")
spark_sql = spark.sql('''
        SELECT cabang, COUNT(*) AS total_transaksi,
        SUM(CASE WHEN kategori_sla = 'Sesuai SLA' THEN 1 ELSE 0 END) AS jumlah_melebihi_sla,
        ROUND(
            (SUM(CASE WHEN kategori_sla = 'Sesuai SLA' THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 2
        ) AS persentase_melebihi_sla        
        FROM gabungan      
        GROUP BY cabang
        ORDER BY persentase_melebihi_sla DESC
    ''')
spark_sql.show()

+-----------------+---------------+-------------------+-----------------------+
|           cabang|total_transaksi|jumlah_melebihi_sla|persentase_melebihi_sla|
+-----------------+---------------+-------------------+-----------------------+
|  Cabang Semarang|            137|                 85|                  62.04|
|Cabang Yogyakarta|            186|                109|                  58.60|
|  Cabang Magelang|             80|                 46|                  57.50|
+-----------------+---------------+-------------------+-----------------------+



In [53]:
# Menyimpan hasil akhir
!hdfs dfs -mkdir -p /user/mazedev/latihan/uts/processed/laporan_transaksi

df_gabungan.write.mode("overwrite").partitionBy("cabang").parquet("hdfs://localhost:9000/user/mazedev/latihan/uts/processed/laporan_transaksi")

!hdfs dfs -ls /user/mazedev/latihan/uts/processed/laporan_transaksi

Found 4 items
-rw-r--r--   3 mazedev supergroup          0 2026-10-08 00:46 /user/mazedev/latihan/uts/processed/laporan_transaksi/_SUCCESS
drwxr-xr-x   - mazedev supergroup          0 2026-10-08 00:46 /user/mazedev/latihan/uts/processed/laporan_transaksi/cabang=Cabang Magelang
drwxr-xr-x   - mazedev supergroup          0 2026-10-08 00:46 /user/mazedev/latihan/uts/processed/laporan_transaksi/cabang=Cabang Semarang
drwxr-xr-x   - mazedev supergroup          0 2026-10-08 00:46 /user/mazedev/latihan/uts/processed/laporan_transaksi/cabang=Cabang Yogyakarta


In [54]:
#Verifikasi
df_parquet = spark.read.parquet("hdfs://localhost:9000/user/mazedev/latihan/uts/processed/laporan_transaksi")

print("Jumlah baris: ", df_parquet.count())

Jumlah baris:  403


In [56]:
spark.stop()